## Exploración y comprensión de los datos:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from PIL import Image

### Cargar el dataset proporcionado y realizar un análisis exploratorio de los datos

In [ ]:
# Dataset principal de crímenes
df = pd.read_csv("Crimes_-_2025_20260312.csv")

# Dataset de Community Areas (nombres de las áreas)
community_areas = pd.read_csv('Boundaries_-_Community_Areas_20260329.csv', usecols=['AREA_NUMBE', 'COMMUNITY'])
community_areas = community_areas.rename(columns={'AREA_NUMBE': 'Community Area', 'COMMUNITY': 'Community Area Description'})
community_areas['Community Area'] = community_areas['Community Area'].astype(int)

# Dataset de códigos IUCR (descripción de delitos)
iucr_codes = pd.read_csv('Chicago_Police_Department_-_Illinois_Uniform_Crime_Reporting_(IUCR)_Codes_20260329.csv')
iucr_codes = iucr_codes.rename(columns={'PRIMARY DESCRIPTION': 'IUCR Primary', 'SECONDARY DESCRIPTION': 'IUCR Secondary', 'INDEX CODE': 'Index Code'})
iucr_codes['IUCR'] = iucr_codes['IUCR'].str.zfill(4)

# Dataset enriquecido: merge único con IUCR y Community Areas
df_chicago = df.merge(iucr_codes[['IUCR', 'IUCR Primary', 'IUCR Secondary', 'Index Code']], on='IUCR', how='left') \
               .merge(community_areas, on='Community Area', how='left')

# Validación: verificar que el merge no duplicó ni perdió filas
print(f"Dataset original:     {df.shape[0]} filas, {df.shape[1]} columnas")
print(f"Dataset enriquecido:  {df_chicago.shape[0]} filas, {df_chicago.shape[1]} columnas")
assert df.shape[0] == df_chicago.shape[0], "ERROR: el merge alteró la cantidad de filas"
print("Validación OK: misma cantidad de filas")
df_chicago

### Describir las características principales del dataset, incluyendo el número de observaciones, número de variables y tipos de datos.

Tipos de variables:
* Localización: latitude, longitude, community area, district, location.
* Tipos de crímenes: IUCR, FBI code, domestic, arrest.
* Claves/ID: ID, case number.

Descripción general
* Filas: 236686
* Columnas: 22
* Columnas ID: 2
* Año: 2025

Variables categóricas
* Date
* Arrest
* Domestic
* FBI Code
* District
* Ward
* Description
* Location

Variables numéricas
* Date
* Latitude
* Longitude

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
df.describe()

In [ ]:
# Asimetría y curtosis de variables numéricas
# Forma de la distribución de variables numéricas:
# Asimetría: 0 = simétrica, >0 = cola derecha, <0 = cola izquierda
# Curtosis: 0 = normal, >0 = colas pesadas (leptocúrtica), <0 = colas livianas (platicúrtica)
cols_num = df.select_dtypes(include=[np.number]).columns
asimetria = df[cols_num].skew()
curtosis = df[cols_num].kurt()

stats_forma = pd.DataFrame({'Asimetría (Skewness)': asimetria, 'Curtosis (Kurtosis)': curtosis})
stats_forma

In [ ]:
# Histogramas con KDE para variables numéricas relevantes
vars_analisis = ['District', 'Ward', 'Community Area', 'Beat']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, col in zip(axes.flatten(), vars_analisis):
    data = df[col].dropna()
    sns.histplot(data, bins=30, kde=True, ax=ax, color='steelblue')
    ax.axvline(data.mean(), color='red', linestyle='--', lw=1.5, label=f'Media: {data.mean():.1f}')
    ax.axvline(data.median(), color='green', linestyle='--', lw=1.5, label=f'Mediana: {data.median():.1f}')
    ax.set_title(f'Distribución de {col} (Skew: {data.skew():.2f}, Kurt: {data.kurt():.2f})')
    ax.legend(fontsize=8)

plt.suptitle('Distribución de Variables Numéricas con KDE', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### Identificar patrones generales y distribuciones.

#### Distribución de Arrestos y Crímenes Domésticos
Se analiza la proporción de arrestos y crímenes domésticos para identificar desbalances en las variables categóricas binarias del dataset.

In [ ]:
# Contar la cantidad de arrestos con porcentajes
arrestos = df_chicago['Arrest'].value_counts()
porcentajes = df_chicago['Arrest'].value_counts(normalize=True) * 100
pd.DataFrame({'Cantidad': arrestos, 'Porcentaje (%)': porcentajes.round(2)})

In [ ]:
# Distribución de Crímenes Domésticos
# En Chicago, el crimen doméstico es cualquier abuso físico, acoso, intimidación, interferencia con la libertad personal o 
# privación voluntaria entre "miembros de la familia o del hogar", definido por la Ley de Violencia Doméstica de Illinois (IDVA). 
# Incluye violencia entre parejas íntimas, cónyuges, ex cónyuges, personas que comparten un hogar, o tienen hijos en común.
# https://www.womenslaw.org/es/leyes/il/ordenes-de-restriccion/ordenes-de-proteccion/informacion-basica/cual-es-la-definicion-legal

domestic_counts = df_chicago['Domestic'].value_counts()
domestic_pct = df_chicago['Domestic'].value_counts(normalize=True) * 100
print("Distribución de Crímenes Domésticos:")
for val, count in domestic_counts.items():
    print(f"  {val}: {count:,} ({domestic_pct[val]:.1f}%)")

In [ ]:
# Relación entre Arrestos y Crímenes Domésticos
cross_tab = pd.crosstab(df_chicago['Domestic'], df_chicago['Arrest'], normalize='index') * 100

fig, ax = plt.subplots(figsize=(10, 6))
x = range(len(cross_tab.index))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], cross_tab[False], width, label='Sin Arresto', color='steelblue')
bars2 = ax.bar([i + width/2 for i in x], cross_tab[True], width, label='Con Arresto', color='coral')

ax.set_xlabel('Crimen Doméstico')
ax.set_ylabel('Porcentaje (%)')
ax.set_title('Relación entre Arrestos y Crímenes Domésticos')
ax.set_xticks(x)
ax.set_xticklabels(['No Doméstico', 'Doméstico'])
ax.legend()

# Agregar etiquetas de porcentaje
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.1f}%', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom')
for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.1f}%', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom')

plt.tight_layout()
plt.show()

In [ ]:
ct = pd.crosstab(df_chicago['Domestic'], df_chicago['Arrest'])

sns.heatmap(ct, annot=True, fmt='d', cmap='Blues')

#### Tipos de crímenes más frecuentes
Se identifican los tipos de crímenes con mayor incidencia y se analiza la tasa de arrestos por código IUCR para detectar qué delitos tienen mayor efectividad policial.

Se utiliza el dataset de [códigos IUCR del Chicago Police Department](https://data.cityofchicago.org/Public-Safety/Chicago-Police-Department-Illinois-Uniform-Crime-R/c7ck-438e/about_data) para enriquecer el análisis con las descripciones oficiales y la clasificación **Index Code**:
- **"I" (Index Crime)**: delitos graves usados por el FBI como indicador de criminalidad (homicidio, robo, agresión agravada, hurto, robo de vehículo, etc.).
- **"N" (Non-Index Crime)**: delitos menores no incluidos en el índice principal (vandalismo, fraude, posesión de drogas, alteración del orden público, etc.).

In [ ]:
# Top 10 tipos de crímenes más frecuentes (Primary Type)
top_crimes = df_chicago['Primary Type'].value_counts().head(10)
print("Top 10 tipos de crímenes más frecuentes:")
for crime, count in top_crimes.items():
    print(f"  {crime}: {count:,}")

In [ ]:
# Histograma de los códigos IUCR más frecuentes
plt.figure(figsize=(14, 6))
iucr_counts = df_chicago['IUCR'].value_counts().head(10)

# Crear etiquetas con Primary Type - Description
etiquetas = []
for iucr in iucr_counts.index:
    info = df_chicago[df_chicago['IUCR'] == iucr][['Primary Type', 'Description']].iloc[0]
    etiquetas.append(f"{info['Primary Type']} - {info['Description']}")

plt.bar(range(len(iucr_counts)), iucr_counts.values)
plt.xticks(range(len(iucr_counts)), etiquetas, rotation=45, ha='right')
plt.title('Top 10 Códigos IUCR más Frecuentes')
plt.xlabel('Tipo de Crimen - Descripción')
plt.ylabel('Cantidad de Crímenes')
plt.tight_layout()
plt.show()

In [ ]:
top_n = 10
top = df_chicago['Primary Type'].value_counts().nlargest(top_n).index

df_top = df_chicago[df_chicago['Primary Type'].isin(top)]

plt.figure(figsize=(12, 6))

ax = sns.countplot(
    data=df_top,
    y='Primary Type',
    order=top,
    palette='viridis'
)

# Porcentajes
total = len(df_top)
for i, p in enumerate(ax.patches):
    value = p.get_width()
    ax.text(value + 5, i, f"{value/total*100:.1f}%", va='center')

plt.title(f'Top {top_n} Primary Type')
plt.xlabel('Cantidad')
plt.ylabel('Primary Type')

plt.tight_layout()
plt.show()

* El crimen de battery (agresión física o lesiones) es el contacto físico intencional, ilícito y no consentido con otra persona que resulta en daño o es ofensivo. A diferencia del asalto (assault), que es la amenaza, la battery implica el contacto físico directo o indirecto (como usar un objeto o arma). 
* El criminal assault (agresión o asalto penal) es un delito que implica un acto intencional que causa en otra persona el temor razonable de sufrir un daño físico o contacto ofensivo inminente. No siempre requiere contacto físico real (lesiones), sino la amenaza creíble de violencia, diferenciándose de la "agresión física" (battery), que es el contacto físico en sí.
* El criminal damage (delito de daños) es un acto criminal que consiste en destruir, dañar o desfigurar intencionalmente o por imprudencia la propiedad ajena sin justificación legal. Puede clasificarse como un delito grave (felonía) si el valor del daño es alto o involucra servicios públicos esenciales.
* El crimen de Deceptive Practice (práctica engañosa) es una categoría amplia de actividad delictiva que implica engañar, estafar o inducir a error a sabiendas e intencionalmente a otra persona, empresa o entidad, generalmente para obtener un beneficio personal, financiero o de propiedad de forma ilícita.
* El burglary (robo con allanamiento de morada o fuerza en las cosas) es un delito penal anglosajón que consiste en entrar ilegalmente en un edificio, vivienda o vehículo cerrado con la intención de cometer un delito en su interior, generalmente robo o un crimen grave. No requiere violencia física contra personas. 
* El delito de robbery (atraco o robo con violencia) es la sustracción ilegal de propiedad ajena utilizando fuerza, amenazas o intimidación directa sobre la víctima. A diferencia del hurto (theft) o allanamiento (burglary), el robbery es considerado un delito violento contra la persona, a menudo castigado con penas severas. 

In [ ]:
# Proporción de delitos índice vs no índice (gráfico simple)
index_pct = df_chicago['Index Code'].value_counts(normalize=True) * 100
index_counts = df_chicago['Index Code'].value_counts()

labels = ['Non-Index (menores)', 'Index (graves)']
sizes = [index_pct.get('N', 0), index_pct.get('I', 0)]
counts = [index_counts.get('N', 0), index_counts.get('I', 0)]
colors = ['steelblue', 'crimson']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
axes[0].pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90, colors=colors)
axes[0].set_title('Proporción Index vs Non-Index')

# Bar chart con cantidades
bars = axes[1].bar(labels, counts, color=colors)
axes[1].set_title('Cantidad de Crímenes por Index Code')
axes[1].set_ylabel('Cantidad')
for bar, count in zip(bars, counts):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                f'{count:,}', ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Proporción de delitos índice vs no índice
index_counts = df_chicago['Index Code'].value_counts()
index_pct = df_chicago['Index Code'].value_counts(normalize=True) * 100

print("Clasificación de crímenes según Index Code (FBI):")
print(f"  Index ('I' - delitos graves): {index_counts.get('I', 0):,} ({index_pct.get('I', 0):.1f}%)")
print(f"  Non-Index ('N' - delitos menores): {index_counts.get('N', 0):,} ({index_pct.get('N', 0):.1f}%)")

# Top 10 crímenes índice vs no índice
top_index = df_chicago[df_chicago['Index Code'] == 'I']['Primary Type'].value_counts().head(10)
top_non_index = df_chicago[df_chicago['Index Code'] == 'N']['Primary Type'].value_counts().head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(top_index.index[::-1], top_index.values[::-1], color='crimson')
axes[0].set_title('Top 10 Delitos Índice (graves)')
axes[0].set_xlabel('Cantidad')

axes[1].barh(top_non_index.index[::-1], top_non_index.values[::-1], color='steelblue')
axes[1].set_title('Top 10 Delitos No Índice (menores)')
axes[1].set_xlabel('Cantidad')

plt.tight_layout()
plt.show()

In [ ]:
# Top tipos de crímenes con mayor tasa de arrestos (por código IUCR)
iucr_total = df_chicago.groupby('IUCR').size()
iucr_arrestos = df_chicago[df_chicago['Arrest'] == True].groupby('IUCR').size()
tasa_arresto = (iucr_arrestos / iucr_total * 100).dropna()

# Filtrar solo IUCR con al menos 50 casos para significancia estadística
iucr_significativos = iucr_total[iucr_total >= 50].index
tasa_arresto_filtrada = tasa_arresto[tasa_arresto.index.isin(iucr_significativos)]
top_tasa_arresto = tasa_arresto_filtrada.sort_values(ascending=False).head(30)

# Etiquetas usando las columnas enriquecidas de df_chicago
etiquetas_arrest = []
for iucr in top_tasa_arresto.index:
    row = df_chicago[df_chicago['IUCR'] == iucr].iloc[0]
    etiquetas_arrest.append(f"{row['IUCR Primary']} - {row['IUCR Secondary']}")

plt.figure(figsize=(14, 6))
bars = plt.bar(range(len(top_tasa_arresto)), top_tasa_arresto.values, color='coral')
plt.xticks(range(len(top_tasa_arresto)), etiquetas_arrest, rotation=45, ha='right')
plt.title('Top Tipos de Crímenes con Mayor Tasa de Arrestos (%)')
plt.xlabel('Tipo de Crimen - Descripción (IUCR)')
plt.ylabel('Tasa de Arrestos (%)')

for bar in bars:
    height = bar.get_height()
    plt.annotate(f'{height:.1f}%', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom')

plt.tight_layout()
plt.show()

In [ ]:
# Top 15 tasa de arrestos por tipo de crimen (IUCR Primary)
tipo_total = df_chicago.groupby('IUCR Primary').size()
tipo_arrestos = df_chicago[df_chicago['Arrest'] == True].groupby('IUCR Primary').size()
tasa = (tipo_arrestos / tipo_total * 100).dropna()

# Filtrar tipos con al menos 50 casos
sig = tipo_total[tipo_total >= 50].index
tasa = tasa[tasa.index.isin(sig)].sort_values(ascending=False).head(15)

plt.figure(figsize=(14, 5))
bars = plt.barh(tasa.index[::-1], tasa.values[::-1], color='coral')
plt.title('Top 15 Tasa de Arrestos por Tipo de Crimen (%)')
plt.xlabel('Tasa de Arrestos (%)')

for bar in bars:
    width = bar.get_width()
    plt.annotate(f'{width:.1f}%', xy=(width, bar.get_y() + bar.get_height()/2),
                xytext=(3, 0), textcoords="offset points", ha='left', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Tasa de arrestos por tipo de crimen: Doméstico vs No Doméstico (por IUCR Primary)
for domestico, titulo, color in [(True, 'Crímenes Domésticos', 'crimson'), (False, 'Crímenes No Domésticos', 'steelblue')]:
    subset = df_chicago[df_chicago['Domestic'] == domestico]
    tipo_total = subset.groupby('IUCR Primary').size()
    tipo_arrestos = subset[subset['Arrest'] == True].groupby('IUCR Primary').size()
    tasa = (tipo_arrestos / tipo_total * 100).dropna()
    
    # Filtrar tipos con al menos 50 casos
    sig = tipo_total[tipo_total >= 50].index
    tasa = tasa[tasa.index.isin(sig)].sort_values(ascending=False).head(15)
    
    plt.figure(figsize=(14, 5))
    bars = plt.barh(tasa.index[::-1], tasa.values[::-1], color=color)
    plt.title(f'Top 15 Tasa de Arrestos por Tipo de Crimen - {titulo} (%)')
    plt.xlabel('Tasa de Arrestos (%)')
    
    for bar in bars:
        width = bar.get_width()
        plt.annotate(f'{width:.1f}%', xy=(width, bar.get_y() + bar.get_height()/2),
                    xytext=(3, 0), textcoords="offset points", ha='left', va='center', fontsize=9)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Tasa de arrestos por tipo de crimen: Doméstico vs No Doméstico
for domestico, titulo, color in [(True, 'Crímenes Domésticos', 'crimson'), (False, 'Crímenes No Domésticos', 'steelblue')]:
    subset = df_chicago[df_chicago['Domestic'] == domestico]
    iucr_total_d = subset.groupby('IUCR').size()
    iucr_arrestos_d = subset[subset['Arrest'] == True].groupby('IUCR').size()
    tasa_d = (iucr_arrestos_d / iucr_total_d * 100).dropna()
    
    # Filtrar IUCR con al menos 30 casos
    sig = iucr_total_d[iucr_total_d >= 30].index
    tasa_d = tasa_d[tasa_d.index.isin(sig)].sort_values(ascending=False).head(15)
    
    etiquetas = []
    for iucr in tasa_d.index:
        row = subset[subset['IUCR'] == iucr].iloc[0]
        etiquetas.append(f"{row['IUCR Primary']} - {row['IUCR Secondary']}")
    
    plt.figure(figsize=(14, 5))
    bars = plt.barh(etiquetas[::-1], tasa_d.values[::-1], color=color)
    plt.title(f'Top 15 Tasa de Arrestos - {titulo} (%)')
    plt.xlabel('Tasa de Arrestos (%)')
    
    for bar in bars:
        width = bar.get_width()
        plt.annotate(f'{width:.1f}%', xy=(width, bar.get_y() + bar.get_height()/2),
                    xytext=(3, 0), textcoords="offset points", ha='left', va='center', fontsize=8)
    
    plt.tight_layout()
    plt.show()

#### Análisis por Community Area
Se determina el crimen prevalente en cada Community Area y se identifican las zonas con mayor cantidad de delitos registrados.

In [ ]:
# Análisis del crimen que prevalece por Community Area
conteo = df_chicago.groupby(['Community Area', 'Primary Type']).size().reset_index(name='Cantidad')
crimen_prevalente = conteo.loc[conteo.groupby('Community Area')['Cantidad'].idxmax()]
crimen_prevalente = crimen_prevalente.rename(columns={'Primary Type': 'Crimen Prevalente'})

# Agregar nombre del Community Area
crimen_prevalente = crimen_prevalente.merge(
    df_chicago[['Community Area', 'Community Area Description']].drop_duplicates(),
    on='Community Area', how='left'
)
crimen_prevalente = crimen_prevalente.sort_values('Cantidad', ascending=False)
crimen_prevalente

In [ ]:
# Top 10 Community Areas con mayor cantidad de crímenes
top_areas = df_chicago.groupby('Community Area').agg(
    Cantidad=('Community Area', 'size'),
    Nombre=('Community Area Description', 'first')
).sort_values('Cantidad', ascending=False).head(10)

print("Top 10 Community Areas con mayor cantidad de crímenes:")
for area, row in top_areas.iterrows():
    print(f"  {row['Nombre']} (Area {int(area)}): {row['Cantidad']:,} crímenes")

plt.figure(figsize=(12, 6))
labels = [f"{row['Nombre']} ({int(area)})" for area, row in top_areas.iterrows()]
plt.barh(labels[::-1], top_areas['Cantidad'].values[::-1], color='teal')
plt.title('Top 10 Community Areas con Mayor Cantidad de Crímenes')
plt.xlabel('Cantidad de Crímenes')
plt.tight_layout()
plt.show()

#### Patrones temporales
Se analiza la distribución de crímenes por día de la semana, hora del día y mes para identificar patrones estacionales y franjas horarias de mayor incidencia.

In [ ]:
# Distribución de crímenes por día de la semana
df_chicago['Date'] = pd.to_datetime(df_chicago['Date'])
df_chicago['DayOfWeek'] = df_chicago['Date'].dt.day_name()

dias_orden = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
casos_por_dia = df_chicago['DayOfWeek'].value_counts().reindex(dias_orden)

plt.figure(figsize=(10, 5))
plt.bar(casos_por_dia.index, casos_por_dia.values, color='steelblue')
plt.title('Cantidad de Crímenes por Día de la Semana')
plt.xlabel('Día')
plt.ylabel('Cantidad')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Distribución de crímenes por hora del día
df_chicago['Hour'] = df_chicago['Date'].dt.hour

casos_por_hora = df_chicago['Hour'].value_counts().sort_index()

plt.figure(figsize=(12, 5))
plt.bar(casos_por_hora.index, casos_por_hora.values, color='coral')
plt.title('Cantidad de Crímenes por Hora del Día')
plt.xlabel('Hora')
plt.ylabel('Cantidad')
plt.xticks(range(0, 24))
plt.tight_layout()
plt.show()

In [ ]:
# Histograma de casos por mes
df_chicago['Date'] = pd.to_datetime(df_chicago['Date'])
df_chicago['Month'] = df_chicago['Date'].dt.month

casos_por_mes = df_chicago['Month'].value_counts().sort_index()

plt.figure(figsize=(10, 6))
plt.bar(casos_por_mes.index, casos_por_mes.values, color='steelblue')
plt.title('Cantidad de Crímenes por Mes')
plt.xlabel('Mes')
plt.ylabel('Cantidad de Crímenes')
plt.xticks(range(1, 13), ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic'])
plt.tight_layout()
plt.show()

Evolución de casos domésitcos por mes

In [ ]:
df_dom = df_chicago[df_chicago["Domestic"] == True].copy()
df_dom["Date"] = pd.to_datetime(df_dom["Date"])

df_dom["mes"] = df_dom["Date"].dt.to_period("M")

df_mes = df_dom.groupby("mes").size().reset_index(name="cantidad")
df_mes = df_mes.sort_values("mes")

# convertir a timestamp para graficar bien
df_mes["mes"] = df_mes["mes"].dt.to_timestamp()

In [ ]:
plt.figure()
plt.plot(df_mes["mes"], df_mes["cantidad"], marker="o")

plt.xlabel("Mes")
plt.ylabel("Cantidad de casos")
plt.title("Evolución mensual de casos Domestic")

plt.xticks(rotation=45)
plt.grid()

plt.show()

Evolución de arrestos por mes

In [ ]:
df_Arrest = df_chicago[df_chicago["Arrest"] == True].copy()
df_Arrest["Date"] = pd.to_datetime(df_Arrest["Date"])

df_Arrest["mes"] = df_Arrest["Date"].dt.to_period("M")

df_mes = df_Arrest.groupby("mes").size().reset_index(name="cantidad")
df_mes = df_mes.sort_values("mes")

# convertir a timestamp para graficar bien
df_mes["mes"] = df_mes["mes"].dt.to_timestamp()

In [ ]:
plt.figure()
plt.plot(df_mes["mes"], df_mes["cantidad"], marker="o")

plt.xlabel("Mes")
plt.ylabel("Cantidad de casos")
plt.title("Evolución mensual de Arrestos")

plt.xticks(rotation=45)
plt.grid()

plt.show()

#### Distribución por tipo de ubicación y correlaciones
Se identifican los lugares más frecuentes donde ocurren crímenes y se analiza la correlación entre las variables numéricas del dataset.

In [ ]:
# Top 10 Location Description (lugares donde más ocurren crímenes)
top_locations = df_chicago['Location Description'].value_counts().head(10)

plt.figure(figsize=(12, 6))
plt.barh(top_locations.index[::-1], top_locations.values[::-1], color='mediumpurple')
plt.title('Top 10 Lugares donde Ocurren Crímenes')
plt.xlabel('Cantidad')
plt.ylabel('Location Description')
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap de correlación entre variables numéricas
cols_numericas = ['District', 'Ward', 'Community Area', 'Arrest', 'Domestic']
df_corr = df_chicago[cols_numericas].copy()
df_corr['Arrest'] = df_corr['Arrest'].astype(int)
df_corr['Domestic'] = df_corr['Domestic'].astype(int)

corr = df_corr.corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.columns)

for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=10)

plt.colorbar(im)
plt.title('Heatmap de Correlación')
plt.tight_layout()
plt.show()

Observaciones del heatmap:
- District y Ward tienen correlación positiva alta (0.66): es esperable ya que las guardias estan asignadas a distritos en la ciudad.
- Community Area tiene correlación negativa con District (-0.48) y Ward (-0.55): esto refleja
  que la numeración de estas divisiones geográficas sigue un orden inverso, no una relación causal.
- Arrest y Domestic no muestran correlación (0.00): la probabilidad de arresto no varía
  significativamente entre crímenes domésticos y no domésticos.
- Arrest no correlaciona con ninguna variable geográfica (~0.00): la tasa de arrestos se
  distribuye de forma uniforme en toda la ciudad, sin concentrarse en zonas específicas.
- Domestic tiene correlación muy baja con Community Area (0.12): los crímenes domésticos
  se distribuyen de forma relativamente homogénea entre las áreas.

Nota: District, Ward y Community Area son identificadores geográficos, no variables ordinales.
Las correlaciones entre ellas reflejan coincidencias en la codificación numérica, no relaciones causales.

### Identificar errores, outliers (anomalías), valores faltantes y su tipo (MCAR, MAR, MNAR).

#### Valores nulos
Se analiza la proporción de valores faltantes en cada columna del dataset, identificando features con alto porcentaje de nulos y los registros afectados.

In [ ]:
# Contar la proporción de valores nulos en cada columna
df_chicago.isna().sum()  

In [ ]:
# Visualización de valores faltantes con missingno
fig, ax = plt.subplots(figsize=(15, 8))
msno.bar(df_chicago, fontsize=12, ax=ax)
plt.title('Completitud de cada columna (barras = datos presentes)')
plt.tight_layout()
plt.show()

In [ ]:
# Matriz de nulidad: patrón de valores faltantes (blanco = NaN)
msno.matrix(df_chicago, figsize=(18, 8), fontsize=12)
plt.title('Matriz de Nulidad - Patrón de valores faltantes')
plt.show()

In [ ]:
# Heatmap de correlación de nulidad: si los NaN de una columna se asocian con los de otra
msno.heatmap(df_chicago, fontsize=12, figsize=(15, 8))
plt.title('Correlación de Nulidad entre columnas')
plt.show()

Analisis de outliers

In [ ]:
# 1. Carga y preparación
# Agrupamos por día para detectar anomalías diarias
df_chicago['Date'] = pd.to_datetime(df_chicago['Date'])
df_chicago_date = df_chicago.set_index('Date')
daily_crimes = df_chicago_date.groupby(df_chicago_date.index.date).size().rename('crime_count').to_frame()
daily_crimes.index = pd.to_datetime(daily_crimes.index)

# 2. Identificación de Outliers usando el método de IQR (Rango Intercuartílico)
Q1 = daily_crimes['crime_count'].quantile(0.25)
Q3 = daily_crimes['crime_count'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Marcamos los outliers
outliers = daily_crimes[(daily_crimes['crime_count'] < lower_bound) | (daily_crimes['crime_count'] > upper_bound)].copy()

# --- GRÁFICO 1: Boxplot Mensual para ver dispersión ---
plt.figure(figsize=(12, 6))
# Añadimos columna de mes para el boxplot
daily_crimes['Month'] = daily_crimes.index.strftime('%B')
sns.boxplot(data=daily_crimes, x='Month', y='crime_count', palette='Set3')
plt.title('Dispersión Diaria de Crímenes por Mes (Detección de Outliers)')
plt.ylabel('Crímenes por día')
plt.xticks(rotation=45)
plt.show()

# --- GRÁFICO 2: Serie temporal con Outliers marcados ---
plt.figure(figsize=(15, 6))
plt.plot(daily_crimes.index, daily_crimes['crime_count'], color='gray', alpha=0.5, label='Frecuencia Diaria')
plt.scatter(outliers.index, outliers['crime_count'], color='red', label='Outliers (Anomalías)', zorder=5)
plt.axhline(upper_bound, color='red', linestyle='--', alpha=0.3, label='Límite Superior')
plt.title('Detección de Anomalías en el Tiempo (2025)')
plt.legend()
plt.show()


In [ ]:
# Features con más del 40% de valores nulos
total_rows = len(df_chicago)
null_percentages = (df_chicago.isna().sum() / total_rows * 100).round(2)
features_high_null = null_percentages[null_percentages > 40].sort_values(ascending=False)
print(f"Features con más del 40% de valores nulos ({len(features_high_null)} features):")
print(features_high_null.to_string())

#### Clasificación de valores faltantes (MCAR, MAR, MNAR)

- **MCAR (Missing Completely At Random)**: la probabilidad de que un dato falte no depende de ninguna variable. El porcentaje de nulos es similar en todos los grupos.
- **MAR (Missing At Random)**: la probabilidad de que un dato falte depende de otra variable observada. El porcentaje de nulos varía según el grupo.
- **MNAR (Missing Not At Random)**: la probabilidad de que un dato falte depende del propio valor faltante. No se puede verificar directamente con los datos.

Para clasificar, se analiza si el porcentaje de nulos en las columnas con más faltantes varía según variables como Primary Type, District o Arrest.


Interpretación:
- Si el % de nulos varía significativamente entre grupos -> MAR (depende de otra variable)
- Si el % de nulos es similar en todos los grupos -> MCAR (aleatorio)

#### Análisis de nulos en `Latitude` / `Longitude` / `Location` (91 nulos, 0,038 %)


In [ ]:
# Registros con valores nulos en coordenadas/ubicación
# Se puede obeservar que son delitos de robo de identidad menores a $300, robos sin armas en un bar, acoso por telefono, o robos menores en la calle.
registros_sin_ubicacion = df_chicago[df_chicago['X Coordinate'].isna() | df_chicago['Y Coordinate'].isna() | 
                             df_chicago['Latitude'].isna() | df_chicago['Longitude'].isna() | 
                             df_chicago['Location'].isna()]
print(f"Total de registros con valores nulos en coordenadas: {len(registros_sin_ubicacion)}")
registros_sin_ubicacion

In [ ]:
def nulos_por_grupo(df, grupo_col, target_col):
    return (
        df.groupby(grupo_col, observed=True)[target_col]
        .apply(lambda x: x.isnull().mean() * 100)
        .reset_index(name=f'% Nulos en {target_col}')
        .sort_values(f'% Nulos en {target_col}', ascending=False)
    )

In [ ]:
# Nulos en Latitude por Primary Type
nulos_lat_tipo = nulos_por_grupo(df_chicago, 'Primary Type', 'Latitude')
print("% de nulos en Latitude por Primary Type (top 10):")
print(nulos_lat_tipo.head(10).to_string(index=False))

In [ ]:
# Nulos en Latitude por District
nulos_lat_distrito = nulos_por_grupo(df_chicago, 'District', 'Latitude')
print("% de nulos en Latitude por District:")
print(nulos_lat_distrito.to_string(index=False))

In [ ]:
# Nulos en Latitude por Arrest
nulos_lat_arresto = nulos_por_grupo(df_chicago, 'Arrest', 'Latitude')
print("% de nulos en Latitude por Arrest:")
print(nulos_lat_arresto.to_string(index=False))

In [ ]:
# Nulos en Latitude por Domestic
nulos_lat_domestic = nulos_por_grupo(df_chicago, 'Domestic', 'Latitude')
print("% de nulos en Latitude por Domestic:")
print(nulos_lat_domestic.to_string(index=False))

Los outputs revelan que los faltantes **no son homogéneos**:

- **Por `Primary Type`**: los delitos sensibles a privacidad concentran más nulos — `DECEPTIVE PRACTICE` (0,35 %), `SEX OFFENSE` (0,23 %), `STALKING` (0,17 %), `OFFENSE INVOLVING CHILDREN` (0,13 %), `CRIMINAL SEXUAL ASSAULT` (0,06 %). Sugiere omisión deliberada de coordenadas para proteger a la víctima.
- **Por `Arrest`**: hay una diferencia muy clara — `Arrest=False` 0,045 % vs `Arrest=True` 0,003 %. Cuando hay arresto, la ubicación queda registrada casi siempre; cuando no, es más probable que falte.
- **Por `District`**: variación moderada (0 % a 0,08 %) sin patrón geográfico marcado — probablemente reflejo de la composición de delitos en cada distrito.
- **Por `Domestic`**: prácticamente parejo (0,040 % vs 0,033 %).

→ **MAR (Missing At Random)**: la ausencia depende de variables observadas (`Primary Type` y `Arrest`), no es puramente aleatoria. La hipótesis más consistente es que ciertos tipos de delito ocultan la ubicación por privacidad de la víctima, y que los casos sin arresto tienen menos verificación de datos. 

#### Análisis de nulos en `Location Description` (1.097 nulos, 0,46 %)

La columna `Location Description` indica el tipo de lugar donde ocurrió el delito (ej.: `STREET`, `APARTMENT`, `BAR OR TAVERN`). Tiene 1.097 registros nulos (0,46 %), mucho más que los 91 de las coordenadas. Se replica el análisis por grupo (`Primary Type`, `District`, `Arrest`, `Domestic`) para determinar si es MCAR, MAR o MNAR.

In [ ]:
# Registros con Location Description nula
registros_sin_location_desc = df_chicago[df_chicago['Location Description'].isna()]
print(f"Total de registros con Location Description nula: {len(registros_sin_location_desc)}")
registros_sin_location_desc[['IUCR', 'Primary Type', 'Description', 'District', 'Arrest', 'Domestic']].head(10)

In [ ]:
# Nulos en Location Description por Primary Type (top 10)
nulos_locdesc_tipo = nulos_por_grupo(df_chicago, 'Primary Type', 'Location Description')
print("% de nulos en Location Description por Primary Type (top 10):")
print(nulos_locdesc_tipo.head(10).to_string(index=False))

In [ ]:
# Nulos en Location Description por District
nulos_locdesc_distrito = nulos_por_grupo(df_chicago, 'District', 'Location Description')
print("% de nulos en Location Description por District:")
print(nulos_locdesc_distrito.to_string(index=False))

In [ ]:
# Nulos en Location Description por Arrest
nulos_locdesc_arrest = nulos_por_grupo(df_chicago, 'Arrest', 'Location Description')
print("% de nulos en Location Description por Arrest:")
print(nulos_locdesc_arrest.to_string(index=False))

In [ ]:
# Nulos en Location Description por Domestic
nulos_locdesc_domestic = nulos_por_grupo(df_chicago, 'Domestic', 'Location Description')
print("% de nulos en Location Description por Domestic:")
print(nulos_locdesc_domestic.to_string(index=False))

#### Summary — `Location Description` (1.097 nulos, 0,46 %)

Los outputs muestran una concentración extrema de los nulos:

- **Por `Primary Type`**: practicamente todos los nulos están en `DECEPTIVE PRACTICE` (7,41 % del total), mientras que el resto de los tipos tienen prácticamente 0 % (`ASSAULT` 0,005 %, `THEFT` 0,004 %, y todos los demás en 0 %). En la muestra de 10 registros, los 10 son `FINANCIAL IDENTITY THEFT` (códigos 1153/1154) — crímenes financieros sin lugar físico claro, lo que explica por qué no se completa el campo.
- **Por `Arrest`**: `Arrest=False` 0,55 % vs `Arrest=True` 0,003 %. Cuando hay arresto, el lugar casi siempre queda registrado.
- **Por `Domestic`**: `Domestic=False` 0,57 % vs `Domestic=True` 0,002 %. Los crímenes domésticos ocurren por definición en un lugar identificable (hogar/familia), por eso nunca faltan.
- **Por `District`**: variación moderada (0 % a 1,14 %, con el distrito 19 en el tope) — probablemente reflejo de la composición de delitos (distritos con más denuncias de fraude tienen más nulos).

→ **MAR (Missing At Random)**: la ausencia depende claramente de variables observadas (`Primary Type`, `Arrest`, `Domestic`). La causa raíz es que los delitos de robo de identidad financiero no ocurren en un lugar físico concreto — no es un error de carga sino una limitación del esquema del dataset para este tipo de delitos. 

#### Análisis de nulos en `IUCR Primary` e `IUCR Secondary` (10.397 nulos, 4,39 %)

Ambas columnas provienen del `left merge` con la tabla `iucr_codes` sobre la clave `IUCR` (ver cell de carga de datos). Por construcción, una fila queda nula en ambas cuando su código `IUCR` no existe en el catálogo. Se replica el análisis por grupo (Primary Type, District, Arrest, Domestic) y además se agrega un agrupamiento por `IUCR` —la propia clave del merge— para confirmar si la ausencia depende de esa variable.

In [ ]:
# Registros con valores nulos en IUCR Primary / IUCR Secondary
registros_sin_iucr = df_chicago[df_chicago['IUCR Primary'].isna() | df_chicago['IUCR Secondary'].isna()]
print(f"Total de registros con valores nulos en IUCR Primary/Secondary: {len(registros_sin_iucr)}")
print(f"¿Los nulos coinciden fila a fila en ambas columnas? "
      f"{(df_chicago['IUCR Primary'].isna() == df_chicago['IUCR Secondary'].isna()).all()}")
registros_sin_iucr[['IUCR', 'Primary Type', 'Description', 'IUCR Primary', 'IUCR Secondary']].head(10)

In [ ]:
# Nulos en IUCR Primary por Primary Type (top 10)
nulos_iucr_tipo = nulos_por_grupo(df_chicago, 'Primary Type', 'IUCR Primary')
print("% de nulos en IUCR Primary por Primary Type (top 10):")
print(nulos_iucr_tipo.head(10).to_string(index=False))

In [ ]:
# Nulos en IUCR Primary por District
nulos_iucr_distrito = nulos_por_grupo(df_chicago, 'District', 'IUCR Primary')
print("% de nulos en IUCR Primary por District:")
print(nulos_iucr_distrito.to_string(index=False))

In [ ]:
# Nulos en IUCR Primary por Arrest
nulos_iucr_arrest = nulos_por_grupo(df_chicago, 'Arrest', 'IUCR Primary')
print("% de nulos en IUCR Primary por Arrest:")
print(nulos_iucr_arrest.to_string(index=False))

In [ ]:
# Nulos en IUCR Primary por Domestic
nulos_iucr_domestic = nulos_por_grupo(df_chicago, 'Domestic', 'IUCR Primary')
print("% de nulos en IUCR Primary por Domestic:")
print(nulos_iucr_domestic.to_string(index=False))

In [ ]:
# Nulos en IUCR Primary por código IUCR (clave del merge) - top 15 con más nulos
# Si aparecen códigos con 100% y 0%, la missingness depende totalmente de IUCR.
nulos_iucr_codigo = nulos_por_grupo(df_chicago, 'IUCR', 'IUCR Primary')
print("% de nulos en IUCR Primary por código IUCR (top 15 con más nulos):")
print(nulos_iucr_codigo.head(15).to_string(index=False))
print(f"\nCódigos IUCR únicos con 100% de nulos: "
      f"{(nulos_iucr_codigo['% Nulos en IUCR Primary'] == 100).sum()}")
print(f"Códigos IUCR únicos con 0% de nulos:   "
      f"{(nulos_iucr_codigo['% Nulos en IUCR Primary'] == 0).sum()}")

#### Summary — `IUCR Primary` / `IUCR Secondary` (10.397 nulos, 4,39 %)

Los outputs confirman que la faltante **depende totalmente de la variable `IUCR`**:

- **Agrupando por código `IUCR`**: 21 códigos únicos tienen 100 % de nulos en `IUCR Primary` (ej.: `1192`, `1197`, `1101`, `1262`, `1263`, `1577`, `1573`, …) y 317 códigos tienen 0 % de nulos. Cada código `IUCR` aparece en el catálogo o no aparece, y ésa es la única variable que determina si los campos `IUCR Primary` / `IUCR Secondary` están poblados.
- **Fila a fila**: los nulos en ambas columnas coinciden al 100 %, como era esperable porque provienen del mismo `left merge`.

→ **MAR (Missing At Random)**: la ausencia depende de una única variable observada (`IUCR`) — no es aleatoria ni depende del propio valor faltante. Es un artefacto del `left merge` con `iucr_codes`: los 21 códigos IUCR del dataset original que no están en la tabla de lookup del Chicago Police Department generan los 10.397 nulos. 

#### Detección de Outliers

En nuestro caso, no tiene sentido aplicar IQR/Z-score a las variables administrativas de ubicación (`District`, `Ward`, `Community Area`, `Beat`): aunque estén codificadas como enteros, son identificadores categóricos. El hecho de que el código de un distrito sea mayor que el de otro no implica ninguna distancia ni magnitud.

Por eso, en esta sección analizamos outliers únicamente sobre `Latitude` y `Longitude` (coordenadas geográficas continuas), pues es útil para detectar errores de geocodificación (puntos fuera del polígono de Chicago, coordenadas en cero, etc.)

Se usan dos métodos clásicos y complementarios:
- IQR (Rango Intercuartílico): `outlier` si `x < Q1 - 1.5·IQR` o `x > Q3 + 1.5·IQR`
- Regla 3σ: `outlier` si `|x - μ| > 3·σ`

In [ ]:
# Detección de outliers en Latitude / Longitude
cols_num = ['Latitude', 'Longitude']
df_num = df_chicago[cols_num].dropna()

# Método IQR
Q1 = df_num.quantile(0.25)
Q3 = df_num.quantile(0.75)
IQR = Q3 - Q1
low_iqr = Q1 - 1.5 * IQR
high_iqr = Q3 + 1.5 * IQR
outliers_iqr = (df_num < low_iqr) | (df_num > high_iqr)

# Método desvíos estándar
mean = df_num.mean()
std = df_num.std(ddof=1)
low_std = mean - 3 * std
high_std = mean + 3 * std
outliers_std = (df_num < low_std) | (df_num > high_std)

n = len(df_num)
print(f"Observaciones analizadas: {n:,}\n")
print(f"{'Variable':<12} {'Método':<8} {'Límite inf.':>14} {'Límite sup.':>14} {'# Outliers':>12} {'% Outliers':>12}")
print("-" * 74)
for col in cols_num:
    print(f"{col:<12} {'IQR':<8} {low_iqr[col]:>14.6f} {high_iqr[col]:>14.6f} "
          f"{outliers_iqr[col].sum():>12,} {outliers_iqr[col].sum() / n * 100:>11.3f}%")
    print(f"{col:<12} {'3σ':<8} {low_std[col]:>14.6f} {high_std[col]:>14.6f} "
          f"{outliers_std[col].sum():>12,} {outliers_std[col].sum() / n * 100:>11.3f}%")

In [ ]:
# Visualización: boxplot por variable + mapa de puntos con outliers IQR resaltados
outlier_any_iqr = outliers_iqr.any(axis=1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Boxplots de Latitude y Longitude
sns.boxplot(y=df_num['Latitude'], ax=axes[0], color='lightblue')
axes[0].set_title('Boxplot — Latitude')

sns.boxplot(y=df_num['Longitude'], ax=axes[1], color='lightgreen')
axes[1].set_title('Boxplot — Longitude')

# Scatter geográfico con outliers resaltados
sns.scatterplot(
    x=df_num['Longitude'], y=df_num['Latitude'],
    hue=outlier_any_iqr.map({False: 'Normal', True: 'Outlier'}),
    palette={'Normal': 'steelblue', 'Outlier': 'red'},
    s=5, alpha=0.3, ax=axes[2], legend='brief'
)
axes[2].set_title('Puntos geográficos — outliers IQR en rojo')
axes[2].set_xlabel('Longitude')
axes[2].set_ylabel('Latitude')

plt.suptitle('Detección visual de outliers — coordenadas geográficas', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Lectura de los resultados — `Latitude` / `Longitude`:**

Sobre **236.595 registros** con coordenadas válidas, los dos métodos coinciden cualitativamente pero con sensibilidades distintas:

- **`Latitude`**: 0 outliers por IQR y 0 por 3σ. La ciudad de Chicago está muy estrechamente confinada en el eje norte–sur (aprox. 41,57°–42,11° según los cortes IQR), así que no aparecen valores extremos en esa dimensión.
- **`Longitude`**: **1.802 outliers por IQR (0,76 %)** y **1.275 por 3σ (0,54 %)**. El IQR es más conservador y marca más casos — coherente con que la distribución de longitudes sea levemente asimétrica, lo que hace que los "bigotes" caigan más adentro. 

La ausencia de outliers en latitud y la presencia sólo en longitud refleja la forma alargada de Chicago de norte a sur, más compacta al este que al oeste.

## Aplicación de técnicas de visualización

Se utilizan técnicas de visualización para ilustrar la distribución geográfica de los crímenes, complementando los gráficos de barras, histogramas y heatmaps presentados en las secciones anteriores.

In [ ]:
# Distribución geográfica de crímenes coloreado por District + mapa de referencia
df_geo = df_chicago.dropna(subset=['Latitude', 'Longitude', 'District'])

fig, axes = plt.subplots(1, 2, figsize=(18, 10), gridspec_kw={'width_ratios': [1, 1.2]})

# Panel izquierdo: mapa de referencia de districts
img = Image.open('Chicago_districts_map.png')
axes[0].imshow(img)
axes[0].set_title('Mapa de Districts de Chicago', fontsize=13)
axes[0].axis('off')

# Panel derecho: scatter plot de crímenes
districts = sorted(df_geo['District'].unique())
colors = plt.cm.tab20(np.linspace(0, 1, len(districts)))

for district, color in zip(districts, colors):
    subset = df_geo[df_geo['District'] == district]
    axes[1].scatter(subset['Longitude'], subset['Latitude'], s=0.3, alpha=0.15,
                    c=[color], label=f'District {int(district)}')

axes[1].set_title('Distribución Geográfica de Crímenes por District', fontsize=13)
axes[1].set_xlabel('Longitud')
axes[1].set_ylabel('Latitud')
axes[1].legend(markerscale=10, fontsize=7, loc='lower left', ncol=2)

plt.tight_layout()
plt.show()

In [ ]:
# Distribución porcentual de crímenes por District (pie chart)
district_counts = df_chicago['District'].value_counts().sort_index()

plt.figure(figsize=(10, 10))
labels = [f'District {int(d)}' for d in district_counts.index]
colors = plt.cm.tab20(np.linspace(0, 1, len(district_counts)))

wedges, texts, autotexts = plt.pie(
    district_counts.values, labels=labels, autopct='%1.1f%%',
    startangle=90, colors=colors, pctdistance=0.85,
    textprops={'fontsize': 8}
)
for autotext in autotexts:
    autotext.set_fontsize(7)

plt.title('Distribución Porcentual de Crímenes por District')
plt.tight_layout()
plt.show()

# Hallazgos del Dataset de Crímenes de Chicago 2025

* Se incluyeron 2 dataset adicionales a los que se hicieron merge, comunnity areas y códigos IUCR
* El dataset tiene 236686 filas y 22 columnas
* Arrestos: Hay un 84% con true vs 16% false
* Crimenes domestics: 80.9% true vs 19.1% false
    * No hay relación entre arrestos y crímenes domésticos
* Delitos Index: 41.8% graves vs 58.2% menores
* Top 10 (gráficos):
    *  Crimenes mas frecuentes: Theft, battery… 
    * Community areas con mayor cantidad de crímenes: Austin, near north side, ..
    * Lugares donde ocurren crimenes: Street, apartment, residence, ..
* Cantidad de crímenes:
    * No hay diferencia entre días de la semana
    * Hay mas crímenes a las 00:00 y hay otro pico en 12:00. Quizás tiene que ver con cambios de guardias o horas donde se cargan
    * Hay una suba de crímenes en el verano de USA
* District y Ward tienen alta correlación 0.66
* Faltantes:
    * X/Y coordinates, Latitude, longitude, location: 91 (0,04% del total)
        * Se puede ver que hay mas nulos en las primary type de deceptive practice (35%), sex offense (22%), stalking (17%), offense involving children (12%)
        * MAR: depende del tipo de variable (primary type y arrest)
    * Location: 1097 (0,46% del total)
        * Prácticamente de estos nulos son por Deceptive Practice, como por ejemplo robo de identidad financiera.
        * Cuando hay arresto practicamente siempre se registra el location
        * No son domésticos, porque ocurren en una ubicación por definición
        * MAR: depende del tipo de delito (primary type, arrest, domestic)
    * IUCR primary/secondary: 10397 (4,34% del total)
        * Son por el merge con el otro dataset. 
        * MAR: depende del IUCR, es porque no están en el dataset que agrega información
* Outliers:
    * Se ve con Longitud que hay outliers para el lado oeste

## Posibles problemas de ML supervisado

A partir del análisis exploratorio realizado, se identifican los siguientes problemas de Machine Learning supervisado que podrían abordarse con este dataset:

| # | Problema | Tipo | Variable Target |
|---|----------|------|----------------|
| 1 | **Predicción de arresto** | Clasificación binaria | `Arrest` (True/False) |
| 2 | **Predicción de crimen doméstico** | Clasificación binaria | `Domestic` (True/False) |
| 3 | **Clasificación del tipo de crimen** | Clasificación multiclase | `Primary Type` |

### Vamos a enfocarnos en predicción de arresto (Propuesta #1)

**Tipo:** Clasificación binaria

**Variable target:** `Arrest` (True / False)

**Justificación:**
- Es una variable binaria clara y bien definida
- Presenta un desbalance natural (~16% True vs ~84% False) que es un desafío interesante y realista
- Tiene aplicación práctica directa: predecir la probabilidad de que un crimen termine en arresto puede ayudar a la asignación de recursos policiales